In [ ]:
import numpy as np

In [ ]:
def replicate_ismi(df_inflation, df_weights, rolling_window=120, k=3):
    """Replication de l'indice ISMI (Lansing & Shapiro, 2026).

    df_inflation : DataFrame (Dates x Catégories) - Taux d'inflation mensuels
    df_weights   : DataFrame (Dates x Catégories) - Poids de dépenses (PCE)
    """
    categories = df_inflation.columns
    dates = df_inflation.index[rolling_window:]

    ismi_series = []
    pos_momentum_series = []
    neg_momentum_series = []

    # Stockage temporaire des chocs (résidus de l'AR(1))
    shocks_dict = {cat: pd.Series(index=df_inflation.index, dtype=float) for cat in categories}

    print("Étape 1 : Estimation des chocs via AR(1) glissant...")
    for t_idx in range(rolling_window, len(df_inflation)):
        current_date = df_inflation.index[t_idx]

        for cat in categories:
            # Fenêtre glissante de 120 mois (Lansing & Shapiro, 2026)
            window_data = df_inflation[cat].iloc[t_idx - rolling_window : t_idx + 1]

            y = window_data.iloc[1:]  # t
            x = window_data.iloc[:-1]  # t-1
            x = sm.add_constant(x)

            # Régression linéaire pour obtenir le choc du mois courant
            model = sm.OLS(y, x).fit()
            # Le dernier résidu correspond au choc du mois t
            shocks_dict[cat].at[current_date] = model.resid.iloc[-1]

    df_shocks = pd.DataFrame(shocks_dict)

    print("Étape 2 : Calcul du momentum et agrégation pondérée...")
    for current_date in dates:
        # Récupérer l'historique récent des chocs pour la règle des k mois consécutifs
        t_pos = df_shocks.index.get_loc(current_date)
        recent_shocks = df_shocks.iloc[t_pos - k + 1 : t_pos + 1]

        # Détection des chocs directionnels consécutifs (ex: 3 mois positifs)
        pos_signal = (recent_shocks > 0).all(axis=0).astype(int)
        neg_signal = (recent_shocks < 0).all(axis=0).astype(int)

        # Extraction des poids pour le mois en cours
        w = df_weights.loc[current_date]
        w_normalized = w / w.sum()

        # Calcul des parts pondérées de momentum positive et négative
        share_pos = np.dot(pos_signal, w_normalized)
        share_neg = np.dot(neg_signal, w_normalized)

        # ISMI = part positive - part négative
        ismi = share_pos - share_neg

        ismi_series.append(ismi)
        pos_momentum_series.append(share_pos)
        neg_momentum_series.append(share_neg)

    results = pd.DataFrame(
        {
            "ISMI": ismi_series,
            "Positive_Momentum": pos_momentum_series,
            "Negative_Momentum": neg_momentum_series,
        },
        index=dates,
    )

    return results